# Tarea 5 – Relevamiento de variables: tipo teórico vs. dtype de Pandas

**Dataset:** Telco Customer Churn · 7043 filas × 21 columnas · **Responsable:** Emmanuel

**Objetivo:** clasificar cada variable según la taxonomía vista en Clase 1 (cualitativa nominal/ordinal, cuantitativa discreta/continua) y contrastarla con el `dtype` que Pandas asigna por defecto al leer el CSV, señalando las discrepancias.

**Clasificación del dataset:** estructurado (tabla con filas y columnas de esquema fijo).

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
try:
    df = pd.read_csv("../../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")  # local
except FileNotFoundError:
    df = pd.read_csv(URL)  # Colab

df.shape

## 1. Tipos de dato que asigna Pandas por defecto

In [ ]:
pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unicos": df.nunique()})

## 2. Tabla de relevamiento

| # | Variable | Tipo teórico | Subtipo | `dtype` Pandas | ¿Coincide? | Observación |
|---|---|---|---|---|---|---|
| 1 | `customerID` | Cualitativa | Nominal (identificador) | `object` | Sí | Código único por cliente (7043 valores distintos). No aporta al análisis: se descarta. |
| 2 | `gender` | Cualitativa | Nominal binaria | `object` | Sí | Female / Male. Conviene pasarla a `category`. |
| 3 | `SeniorCitizen` | Cualitativa | Nominal binaria | `int64` | **No** | Es una categoría (0 = no, 1 = sí) guardada como número. Calcular su media sería poco interpretable; hay que tratarla como categórica. |
| 4 | `Partner` | Cualitativa | Nominal binaria | `object` | Sí | Yes / No. |
| 5 | `Dependents` | Cualitativa | Nominal binaria | `object` | Sí | Yes / No. |
| 6 | `tenure` | Cuantitativa | Discreta | `int64` | Sí | Meses de antigüedad, de 0 a 72 (73 valores distintos). |
| 7 | `PhoneService` | Cualitativa | Nominal binaria | `object` | Sí | Yes / No. |
| 8 | `MultipleLines` | Cualitativa | Nominal (3 categorías) | `object` | Sí | No / Yes / "No phone service". La tercera categoría es redundante con `PhoneService = No`. |
| 9 | `InternetService` | Cualitativa | Nominal (3 categorías) | `object` | Sí | DSL / Fiber optic / No. |
| 10 | `OnlineSecurity` | Cualitativa | Nominal (3 categorías) | `object` | Sí | Yes / No / "No internet service" (redundante con `InternetService = No`). |
| 11 | `OnlineBackup` | Cualitativa | Nominal (3 categorías) | `object` | Sí | Ídem 10. |
| 12 | `DeviceProtection` | Cualitativa | Nominal (3 categorías) | `object` | Sí | Ídem 10. |
| 13 | `TechSupport` | Cualitativa | Nominal (3 categorías) | `object` | Sí | Ídem 10. |
| 14 | `StreamingTV` | Cualitativa | Nominal (3 categorías) | `object` | Sí | Ídem 10. |
| 15 | `StreamingMovies` | Cualitativa | Nominal (3 categorías) | `object` | Sí | Ídem 10. |
| 16 | `Contract` | Cualitativa | **Ordinal** | `object` | **No** | Tiene orden natural (Month-to-month < One year < Two year), pero Pandas la lee como texto sin orden. Se declara con `pd.Categorical(..., ordered=True)` o `OrdinalEncoder`. |
| 17 | `PaperlessBilling` | Cualitativa | Nominal binaria | `object` | Sí | Yes / No. |
| 18 | `PaymentMethod` | Cualitativa | Nominal (4 categorías) | `object` | Sí | Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic). |
| 19 | `MonthlyCharges` | Cuantitativa | Continua | `float64` | Sí | Cargo mensual en moneda. |
| 20 | `TotalCharges` | Cuantitativa | Continua | `object` | **No** | Es un monto, pero Pandas la lee como texto porque 11 filas contienen un espacio `" "` (todas con `tenure = 0`). Hay que convertirla con `pd.to_numeric(..., errors="coerce")` (tarea 6). |
| 21 | `Churn` | Cualitativa | Nominal binaria (**variable objetivo**) | `object` | Sí | Yes / No (26,5 % de Yes). |

> Los dtypes de la tabla son los que devuelve `pd.read_csv` por defecto; la sección 1 los verifica al ejecutar el notebook.

## 3. Resumen de discrepancias

| Variable | Pandas asigna | Debería ser | Acción (tarea que la resuelve) |
|---|---|---|---|
| `TotalCharges` | `object` | `float64` (cuantitativa continua) | Convertir con `to_numeric` y tratar los 11 vacíos (6) |
| `SeniorCitizen` | `int64` | categórica binaria | Convertir a `category` o Yes/No (7) |
| `Contract` | `object` | categórica **ordinal** | Declarar el orden o usar `OrdinalEncoder` (7 y 22) |

**Conclusión:** de las 21 variables, **18** tienen un `dtype` coherente con su naturaleza estadística y **3** requieren corrección. Además, las 16 variables categóricas que Pandas lee como texto (todas las `object` salvo `customerID` y `TotalCharges`) conviene pasarlas a `category`: ahorra memoria y deja explícito que son categóricas.

**Cantidades:** 3 cuantitativas (`tenure` discreta; `MonthlyCharges` y `TotalCharges` continuas) y 18 cualitativas (1 ordinal, 16 nominales y 1 identificador, `customerID`, que no se analiza).